# KNN and Cross-Validation

This notebook is intentionally self-contained. It does not import local utility files, so the statistical idea, simulation, Python functions, evaluation, plots, and exam translation are all visible in one place.

## What problem are we solving?

KNN classifies a point by the majority class among its nearest training neighbors.

**Highest-probability exam pattern:** Use stratified CV to choose k and explain the bias-variance tradeoff.

## Assumptions, intuition, and theory

Small k is flexible and can overfit noisy labels. Large k is smoother and can underfit nonlinear boundaries. KNN is distance-based, so scaling matters when variables have different units.

## Python method notes and documentation

`KNeighborsClassifier` stores the training data, `n_neighbors` controls complexity, and `cross_val_score` estimates accuracy for each candidate k.

Documentation links:

- [NumPy random generator](https://numpy.org/doc/stable/reference/random/generator.html)
- [pandas DataFrame](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html)
- [matplotlib pyplot](https://matplotlib.org/stable/api/pyplot_summary.html)
- [KNeighborsClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html)
- [StratifiedKFold](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedKFold.html)
- [cross_val_score](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html)
- [make_classification](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_classification.html)
- [make_moons](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_moons.html)
- [make_blobs](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_blobs.html)

## Fully self-contained worked example

Every code line below is commented so it is easy to scan under exam time pressure.

In [ ]:
import numpy as np  # Import numerical arrays and random-number tools.
import pandas as pd  # Import DataFrame tables for k tuning.
import matplotlib.pyplot as plt  # Import plotting tools for accuracy curves.
from sklearn.datasets import make_classification, make_moons  # Import classification simulators.
from sklearn.metrics import accuracy_score  # Import accuracy metric.
from sklearn.model_selection import StratifiedKFold, cross_val_score  # Import stratified CV tools.
from sklearn.neighbors import KNeighborsClassifier  # Import KNN classifier.
RANDOM_SEED = 123  # Store the reproducibility seed.
np.random.seed(RANDOM_SEED)  # Fix legacy NumPy randomness.
plt.style.use('default')  # Use a predictable plotting style.


In [ ]:
def make_moon_classification_data(n=200, noise=0.25, random_state=123):  # Define a nonlinear classification simulator.
    X, y = make_moons(  # Generate interlocking two-moon classes.
        n_samples=n,  # Set the number of observations.
        noise=noise,  # Add realistic class overlap.
        random_state=random_state,  # Fix the simulation seed.
    )  # Finish the simulator call.
    return X, y  # Return predictors and labels.

def choose_knn_k_cv(X, y, k_values, cv=5, random_state=123):  # Define a local KNN tuning helper.
    splitter = StratifiedKFold(n_splits=cv, shuffle=True, random_state=random_state)  # Create stratified CV folds.
    rows = []  # Create a list for candidate-k results.
    for k in k_values:  # Loop through possible neighbor counts.
        model = KNeighborsClassifier(n_neighbors=k)  # Create a KNN classifier for this k.
        scores = cross_val_score(model, X, y, scoring='accuracy', cv=splitter)  # Estimate accuracy by CV.
        rows.append({'k': k, 'mean_accuracy': np.mean(scores), 'std_accuracy': np.std(scores, ddof=1)})  # Store mean and variability.
    results = pd.DataFrame(rows)  # Convert results to a DataFrame.
    best_k = int(results.loc[results['mean_accuracy'].idxmax(), 'k'])  # Choose k with highest mean CV accuracy.
    return results, best_k  # Return the full table and selected k.


In [ ]:
X, y = make_moon_classification_data(n=200, noise=0.25, random_state=RANDOM_SEED)  # Simulate nonlinear class boundaries.
k_values = range(1, 32, 2)  # Use odd k values to reduce tie risk.
results, best_k = choose_knn_k_cv(X, y, k_values, cv=5, random_state=RANDOM_SEED)  # Tune k by stratified CV.
display(results)  # Display the tuning table.
print(f'Best k by CV accuracy: {best_k}')  # Print the selected k.
plt.figure(figsize=(6, 4))  # Create an accuracy-curve figure.
plt.plot(results['k'], results['mean_accuracy'], marker='o')  # Plot mean CV accuracy by k.
plt.xlabel('number of neighbors k')  # Label the tuning axis.
plt.ylabel('CV accuracy')  # Label the performance axis.
plt.title('KNN tuning by cross-validation')  # Title the plot.
plt.show()  # Render the tuning plot.


## Exam-style problem prompt

A prompt gives a classification data set and asks you to choose KNN k. Compute CV accuracy over odd k values and choose the largest accuracy.

## Adaptable code pattern

```python
# Step 1: Create candidate k values.
# Step 2: Use StratifiedKFold for classification CV.
# Step 3: Fit KNN inside each fold.
# Step 4: Average validation accuracy for every k.
# Step 5: Choose the k with best CV accuracy.
# Step 6: Explain small k as flexible and large k as smooth.
```

## What to remember for the exam

KNN has almost no training phase, but prediction depends heavily on distance and k. Tune k; do not pick it by appearance alone.